# Experiment 4: Optuna Best Config — MLP HP Search

**Single variable changed**: HPs switched from Phase 1 defaults to Optuna-discovered best values.
**Held constant**: architecture family (MLP), loss (CrossEntropy), data (no augmentation).

## Best config from Optuna search

| HP | Optuna best | Phase 1 default |
|----|:-----------:|:---------------:|
| n_layers | 3 | 2 |
| units_0 | 512 | 256 |
| units_1 | 256 | 128 |
| units_2 | 128 | — |
| activation | GELU | ReLU |
| dropout | 0.25 | 0.2 |
| lr | 0.003 | 0.001 |
| weight_decay | 1e-05 | 0 |
| optimizer | AdamW | Adam |
| scheduler | cosine | CosineAnnealingLR |

Includes logit bias sweep (same as Phase 1 A3) for direct comparison.


In [1]:
import sys, os

def _find_root(marker="src", max_up=3):
    p = os.path.abspath(os.getcwd())
    for _ in range(max_up + 1):
        if os.path.isdir(os.path.join(p, marker)):
            return p
        p = os.path.dirname(p)
    return os.path.abspath(os.getcwd())
PROJ_ROOT = _find_root()
sys.path.insert(0, PROJ_ROOT)

import torch, torch.nn as nn, torch.optim as optim
import numpy as np
import torchvision.transforms as transforms

from torch.optim.lr_scheduler import CosineAnnealingLR, StepLR
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score
from src.train_utils import train_one_epoch
from src.eval_utils import (
    evaluate_detailed, get_all_probas_and_labels,
    compute_roc_auc_scores, compute_pr_auc_scores
)

OUT_DIR = os.path.join(PROJ_ROOT, 'outputs/error_analysis/MLP/optuna_best')
os.makedirs(OUT_DIR, exist_ok=True)

DATA_DIR = os.path.join(PROJ_ROOT, 'data')

print(f'PyTorch version: {torch.__version__}')
if torch.backends.mps.is_available():
    device = torch.device('mps')
elif torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')
print(f'Using device: {device}')
print(f'OUT_DIR: {OUT_DIR}')


PyTorch version: 2.13.0+cu130
Using device: cuda
OUT_DIR: c:\document\Study documents\Deeplearning_Course\outputs/error_analysis/MLP/optuna_best


## Dataset — identical to Phase 1


In [2]:
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
train_ds = __import__('torchvision').datasets.FashionMNIST(root=DATA_DIR, train=True, download=True, transform=transform)
test_ds = __import__('torchvision').datasets.FashionMNIST(root=DATA_DIR, train=False, download=True, transform=transform)
class_names = train_ds.classes

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False)
print(f'Train: {len(train_loader)} batches  Test: {len(test_loader)} batches')


Train: 938 batches  Test: 40 batches


## Model — MLP with Optuna best config


In [3]:
class MLPBest(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
                nn.Linear(784, 512),
                nn.GELU(),
                nn.Dropout(0.25),
                nn.Linear(512, 256),
                nn.GELU(),
                nn.Dropout(0.25),
                nn.Linear(256, 128),
                nn.GELU(),
                nn.Dropout(0.25),
                nn.Linear(128, 10),
        )
    def forward(self, x): return self.net(x)

model = MLPBest().to(device)
print(f'Params: {sum(p.numel() for p in model.parameters()):,}')


Params: 567,434


c:\document\Study documents\Deeplearning_Course\.venv\Lib\site-packages\torch\nn\modules\module.py:1369: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\c10/cuda/CUDAAllocatorConfig.h:40.)
  return t.to(


## Training — Optuna best HPs, 30 epochs


In [4]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=0.003, weight_decay=1e-05)
scheduler = CosineAnnealingLR(optimizer, T_max=30)
EPOCHS = 30

train_losses = []
model.train()
for epoch in range(EPOCHS):
    loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    train_losses.append(loss)
    if scheduler: scheduler.step()
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f'Epoch [{epoch+1}/{EPOCHS}] Loss: {loss:.4f}')

torch.save(model.state_dict(), os.path.join(OUT_DIR, 'model_weights.pth'))
with open(os.path.join(OUT_DIR, 'train_losses.txt'), 'w') as f:
    for l in train_losses: f.write(f'{l}\n')
print(f'Done. Final loss: {train_losses[-1]:.4f}')


Epoch [1/30] Loss: 0.6174
Epoch [5/30] Loss: 0.4579


KeyboardInterrupt: 

## Evaluation (bias=0)


In [ ]:
accuracy, cm, per_class = evaluate_detailed(model, test_loader, device, class_names, model_name='OptunaMLP')
cm_np = cm.cpu().numpy()

with open(os.path.join(OUT_DIR, 'metrics_summary.txt'), 'w') as f:
    f.write(f'Test Accuracy (percentage): {accuracy:.2f}\n')
    f.write(f'Test Accuracy (fraction): {accuracy / 100:.4f}\n\n')
    f.write('Macro PR-AUC: PENDING\n\n')
    f.write(f'{"Class":<15} {"TPR":>8} {"Precision":>10}\n')
    f.write('-' * 33 + '\n')
    for i, name in enumerate(class_names):
        tpr = per_class[name]['TPR']; prec = per_class[name]['Precision']
        f.write(f'{name:<15} {tpr:>8.4f} {prec:>10.4f}\n')

with open(os.path.join(OUT_DIR, 'confusion_matrix.txt'), 'w') as f:
    f.write(f'{"":>15}')
    for name in class_names: f.write(f'{name:>15}')
    f.write('\n')
    for i in range(10):
        f.write(f'{class_names[i]:>15}')
        for j in range(10): f.write(f'{cm_np[i, j]:>15}')
        f.write('\n')

with open(os.path.join(OUT_DIR, 'misclassification_analysis.txt'), 'w') as f:
    f.write('Misclassification Analysis\n' + '=' * 70 + '\n\n')
    for c in range(10):
        name = class_names[c]; errors = cm_np[c].sum() - cm_np[c, c]
        f.write(f'True: {name} (errors: {errors})\n' + '-' * 50 + '\n')
        for p in np.argsort(-cm_np[c]):
            if p == c or cm_np[c, p] == 0: continue
            f.write(f'  -> {class_names[p]:<15} count={cm_np[c, p]:>4}\n')
        f.write('\n')

print(f'Bias=0: Acc={accuracy:.2f}%  Shirt TPR={per_class["Shirt"]["TPR"]:.4f}  Prec={per_class["Shirt"]["Precision"]:.4f}')


## Logit bias sweep (same as Phase 1 A3)


In [ ]:
SHIRT_IDX = 6
BIASES = [-1.0, -0.5, 0.0, 0.5, 1.0, 1.5, 2.0]
sweep = []

model.eval()
with torch.no_grad():
    for bias in BIASES:
        all_p, all_l = [], []
        sh_tp = sh_fp = sh_fn = 0
        for inputs, lbls in test_loader:
            inputs, lbls = inputs.to(device), lbls.to(device)
            logits = model(inputs)
            logits[:, SHIRT_IDX] += bias
            preds = logits.argmax(dim=1)
            all_p.extend(preds.cpu().numpy()); all_l.extend(lbls.cpu().numpy())
            for true, pred in zip(lbls.cpu().numpy(), preds.cpu().numpy()):
                if pred == SHIRT_IDX and true == SHIRT_IDX: sh_tp += 1
                if pred == SHIRT_IDX and true != SHIRT_IDX: sh_fp += 1
                if pred != SHIRT_IDX and true == SHIRT_IDX: sh_fn += 1
        acc = accuracy_score(all_l, all_p)
        sweep.append({'bias': bias, 'acc': round(acc*100, 2), 'tpr': round(sh_tp/(sh_tp+sh_fn+1e-8), 4), 'prec': round(sh_tp/(sh_tp+sh_fp+1e-8), 4)})
        print(f'bias={bias:+.1f}  acc={acc*100:.2f}%  Shirt TPR={sh_tp/(sh_tp+sh_fn+1e-8):.4f}  Prec={sh_tp/(sh_tp+sh_fp+1e-8):.4f}')

with open(os.path.join(OUT_DIR, 'bias_sweep_results.txt'), 'w') as f:
    f.write(f'{"Bias":>6} {"Acc%":>7} {"ShirtTPR":>9} {"ShirtPrec":>10}\n' + '-' * 32 + '\n')
    for r in sweep:
        f.write(f'{r["bias"]:>+5.1f} {r["acc"]:>7.2f} {r["tpr"]:>9.4f} {r["prec"]:>10.4f}\n')


## Comparison vs Phase 1 baseline, Wider, and Deeper


In [ ]:
bt = max(sweep, key=lambda r: r['acc'] + r['tpr'] * 100)
z = [r for r in sweep if r['bias'] == 0.0][0]

print(f'{"Config":<30} {"Acc%":>7} {"ShirtTPR":>9} {"ShirtPrec":>10}')
print('-' * 56)
print(f'{"Phase 1 baseline":<30} {"90.08":>7} {"0.7070":>9} {"0.7505":>10}')
print(f'{"Exp 2 Wider":<30} {"90.26":>7} {"0.7280":>9} {"0.7599":>10}')
print(f'{"Exp 3 Deeper":<30} {"90.13":>7} {"0.7140":>9} {"0.7645":>10}')
print(f'{"Exp 4 optuna bias=0":<30} {z["acc"]:>7.2f} {z["tpr"]:>9.4f} {z["prec"]:>10.4f}')
print(f'{"Exp 4 bias="+str(bt["bias"])+" (best trade)":<30} {bt["acc"]:>7.2f} {bt["tpr"]:>9.4f} {bt["prec"]:>10.4f}')
